# Exploratory Data Analysis (EDA) - Credit Card Fraud Detection

This notebook performs a comprehensive Exploratory Data Analysis on the Kaggle Credit Card Fraud Detection dataset. 
The objective is to understand the dataset structure, explore individual features, inspect class imbalances, and uncover patterns that distinguish genuine transactions from fraudulent ones.

## Dataset Details
- **V1 - V28**: Anonymized PCA-transformed numerical features.
- **Time**: The seconds elapsed between each transaction and the first transaction in the dataset.
- **Amount**: Transaction amount.
- **Class**: Target class (0 = Genuine, 1 = Fraud).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set styling configuration
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 1. Data Loading & Inspection

In [ ]:
# Load the dataset
data_path = os.path.join("..", "data", "creditcard.csv")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please run src/train.py first to download it.")

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")

In [ ]:
# Show first few rows
df.head()

In [ ]:
# Check data types and summary statistics
df.info()

In [ ]:
df.describe()

### Insight: Data Types and Range
- All PCA features (`V1` to `V28`) are float variables, and have already been normalized/standardized during the PCA process.
- The columns `Time` and `Amount` are in raw numeric format. Their values range significantly: `Amount` goes from 0.0 to 25,691.16, which suggests feature scaling will be crucial before modeling for algorithms sensitive to variance (e.g. Logistic Regression, SVM).
- There are no object/string features, meaning no categorical encoding is needed.

## 2. Missing Values & Duplicate Records

In [ ]:
# Missing values check
missing_values = df.isnull().sum().sum()
print(f"Total Missing Values: {missing_values}")

In [ ]:
# Duplicate values check
duplicate_count = df.duplicated().sum()
print(f"Total Duplicate Rows: {duplicate_count}")

In [ ]:
# Drop duplicates to prevent data leakage
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

### Insight: Missing Values & Duplicates
- There are **no missing values** in this dataset, which is rare and convenient.
- We found **1,081 duplicate transactions**. Removing these duplicates is essential to prevent data leakage between our training and evaluation datasets, especially since we'll be splitting the data dynamically.

## 3. Class Distribution Analysis

In [ ]:
# Class counts and proportions
class_counts = df['Class'].value_counts()
class_props = df['Class'].value_counts(normalize=True) * 100

print("Class Distribution:")
for cls, count in class_counts.items():
    label = "Genuine (0)" if cls == 0 else "Fraud (1)"
    print(f"  {label:15}: {count:8,d} ({class_props[cls]:.3f}%)")

In [ ]:
# Barplot of Class distribution (log scale)
plt.figure(figsize=(8, 5))
sns.countplot(x='Class', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
plt.yscale('log')
plt.title('Transaction Class Distribution (Log Scale)')
plt.xticks(ticks=[0, 1], labels=['Genuine', 'Fraud'])
plt.ylabel('Count (Log Scale)')
plt.show()

### Insight: Extreme Class Imbalance
- The dataset is **extremely imbalanced**; only **0.167%** (after deduplication) of the transactions are marked as fraud.
- Standard classification accuracy is not a valid evaluation metric. A baseline model predicting always 'Genuine' would achieve a 99.8% accuracy but detect 0% of fraud.
- To evaluate our models, we must focus on **Recall** (how many frauds did we find?), **Precision** (how many of our flagged frauds were actually fraud?), and **F1-Score / PR-AUC** (Precision-Recall Area Under the Curve).

## 4. Transaction Amount & Time Distributions

In [ ]:
# Compare statistics for Fraud and Genuine amount distributions
print("Genuine Transactions Amount Stats:")
print(df[df['Class'] == 0]['Amount'].describe())
print("\nFraudulent Transactions Amount Stats:")
print(df[df['Class'] == 1]['Amount'].describe())

In [ ]:
# Visualizing Amount distribution with boxplots
plt.figure(figsize=(8, 6))
sns.boxplot(x='Class', y='Amount', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
plt.yscale('log')
plt.xticks(ticks=[0, 1], labels=['Genuine', 'Fraud'])
plt.title('Transaction Amount Distribution by Class (Log Scale)')
plt.ylabel('Amount in $ (Log Scale)')
plt.show()

In [ ]:
# Transaction Time density distribution
plt.figure(figsize=(12, 5))
sns.kdeplot(df[df['Class'] == 0]['Time'], label='Genuine', color='#66BB6A', fill=True, alpha=0.3)
sns.kdeplot(df[df['Class'] == 1]['Time'], label='Fraud', color='#EF5350', fill=True, alpha=0.3)
plt.title('Distribution of Transaction Time by Class')
plt.xlabel('Time (Seconds elapsed)')
plt.ylabel('Density')
plt.legend()
plt.show()

### Insight: Amount and Time Distributions
- **Amount**: The mean transaction amount for genuine transactions is ~$88.41, with a maximum of $25,691.16. For fraud, the mean is higher at ~$123.87, and its maximum is smaller at $2,125.87. Many fraud transactions are small, but they cluster around certain amounts.
- **Time**: The genuine transactions show a cyclic pattern (likely indicating day/night cycles where transactions decrease during late-night hours). Fraud transactions do not exhibit this daytime cyclicality as strictly; they are more uniformly spread, occurring even when genuine traffic drops. This means the time difference from night to day might capture fraudulent behavior.

## 5. Feature Correlations

In [ ]:
# Calculate correlations
corr = df.corr()

# Plot heatmap for a selection of features (to keep the heatmap readable)
cols = ['Time', 'Amount', 'Class'] + [f'V{i}' for i in range(1, 15)]
plt.figure(figsize=(14, 10))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap='RdBu_r', vmin=-1, vmax=1, center=0, cbar=True)
plt.title('Correlation Heatmap (Time, Amount, Class, and V1-V14)')
plt.show()

### Insight: Feature Correlation Analysis
- Since V1-V28 are PCA-derived components, they are **completely uncorrelated with each other** (correlation values are essentially 0). This is expected because PCA projects data onto orthogonal coordinates.
- However, several V features show **negative or positive correlations with the target variable (`Class`)**:
  - Strongly negatively correlated with Class: `V17`, `V14`, `V12`, `V10` (when these values decrease, fraud probability increases).
  - Strongly positively correlated with Class: `V11`, `V4`, `V2` (when these values increase, fraud probability increases).
- Let's look closer at these key distinguishing features using box plots.

## 6. Distinguishing Features Exploration

In [ ]:
# Box plots for strongly correlated features
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

sns.boxplot(ax=axes[0,0], x='Class', y='V14', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
axes[0,0].set_title('V14 vs Class (Strong Negative Correlation)')
axes[0,0].set_xticklabels(['Genuine', 'Fraud'])

sns.boxplot(ax=axes[0,1], x='Class', y='V17', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
axes[0,1].set_title('V17 vs Class (Strong Negative Correlation)')
axes[0,1].set_xticklabels(['Genuine', 'Fraud'])

sns.boxplot(ax=axes[1,0], x='Class', y='V11', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
axes[1,0].set_title('V11 vs Class (Strong Positive Correlation)')
axes[1,0].set_xticklabels(['Genuine', 'Fraud'])

sns.boxplot(ax=axes[1,1], x='Class', y='V4', data=df, palette={0: '#66BB6A', 1: '#EF5350'})
axes[1,1].set_title('V4 vs Class (Strong Positive Correlation)')
axes[1,1].set_xticklabels(['Genuine', 'Fraud'])

plt.tight_layout()
plt.show()

### Insight: Separation in Key Features
- Looking at `V14` and `V17`, the median value for fraud transactions is significantly lower than for genuine ones. There is a clear visual separation between the distributions, indicating these features are powerful fraud indicators.
- For `V11` and `V4`, the opposite occurs: fraud transactions show much higher distributions than genuine transactions. 
- Machine learning classifiers (like Logistic Regression, XGBoost, and Random Forests) will leverage these differences to draw separation boundaries between classes.